## Comparative analysis of results in Bologna:

### Mean Radiant Temperature predicted from High resolution data and open Global datasets

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib.ticker as mticker
from rasterio.transform import rowcol
import pandas as pd
import geopandas as gpd
from rasterio.features import geometry_mask
from rasterio.windows import Window, from_bounds
from matplotlib.colors import ListedColormap

In [ ]:
# inputs
mrt_predicted_hr_data = '../../datos-notebooks/3.HR-data-bologna/MRT-prediction/MRT_1m_condistance.tif'
mrt_predicted_global_data_2 = '../../datos-notebooks/2.global-data-bologna/MRT-prediction/MRT_fromglobal_1m.tif'

# shadow inputs (buildings and trees heights)
buildings_hr = '../../datos-notebooks/3.HR-data-bologna/objects-height/rasterized-0.5m/buildings_height_0.5m.tif'
trees_hr = '../../datos-notebooks/3.HR-data-bologna/objects-height/rasterized-0.5m/trees_height_0.5m.tif'
buildings_global = '../../datos-notebooks/2.global-data-bologna/objects-height/rasterized-1m/buildings_height_1m.tif'  # 0.5 m despite the name
trees_global = '../../datos-notebooks/2.global-data-bologna/objects-height/rasterized-1m/trees_height_1m.tif'
city_boundary = '../../datos-notebooks/3.HR-data-bologna/base-data/vectors/bologne_boundary.geojson'

# outputs
output_folder = '../../datos-notebooks/4.comparative-analysis/'
os.makedirs(output_folder, exist_ok=True)
out_diff      = os.path.join(output_folder, 'MRT_difference_bologna.tif')
out_shadow_inputs_csv = os.path.join(output_folder, 'shadow_inputs_metrics.csv')

### differences between predictions 

In [ ]:
with  rasterio.open(mrt_predicted_hr_data) as hr, rasterio.open(mrt_predicted_global_data_2) as glob:
    xmin, ymin, xmax, ymax = hr.bounds
    hr_data = hr.read(1).astype("float32")  
    glob_data = glob.read(1).astype("float32")
    
fig, axes = plt.subplots(1, 2, figsize=(8, 4))



# HR predicted
im1 = axes[0].imshow(hr_data, cmap="inferno", vmin=np.nanmin(hr_data), vmax=np.nanmax(hr_data))
axes[0].set_title("MRT estimada\n(datos HR)")
plt.colorbar(im1, ax=axes[0], fraction=0.046)

# Global predicted
im2 = axes[1].imshow(glob_data, cmap="inferno", vmin=np.nanmin(hr_data), vmax=np.nanmax(hr_data))
axes[1].set_title("MRT estimada\n(datos globales)")
plt.colorbar(im2, ax=axes[1], fraction=0.046)



for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
#window


# Define the window in geographic coordinates (adjust these values)
win_xmin, win_ymin, win_xmax, win_ymax = 686000, 4928000, 687000, 4929000

with rasterio.open(mrt_predicted_hr_data) as hr, rasterio.open(mrt_predicted_global_data_2) as glob:
    transform = hr.transform
    
    # Convert geographic coordinates to pixel indices
    row_min, col_min = rowcol(transform, win_xmin, win_ymax)  # upper-left corner
    row_max, col_max = rowcol(transform, win_xmax, win_ymin)  # lower-right corner
    
    hr_data   = hr.read(1).astype("float32")[row_min:row_max, col_min:col_max]
    glob_data = glob.read(1).astype("float32")[row_min:row_max, col_min:col_max]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

im1 = axes[0].imshow(hr_data, cmap="inferno", vmin=np.nanmin(hr_data), vmax=np.nanmax(hr_data))
axes[0].set_title("MRT estimada\n(datos HR)")
plt.colorbar(im1, ax=axes[0], fraction=0.046)

im2 = axes[1].imshow(glob_data, cmap="inferno", vmin=np.nanmin(hr_data), vmax=np.nanmax(hr_data))
axes[1].set_title("MRT estimada\n(datos globales)")
plt.colorbar(im2, ax=axes[1], fraction=0.046)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

#### model trained with HR data applied with HR data - model trained with global applied with global datasets

In [ ]:
os.makedirs(os.path.dirname(out_diff), exist_ok=True)

with rasterio.open(mrt_predicted_hr_data) as src1, \
     rasterio.open(mrt_predicted_global_data_2) as src2:

    # determine common intersection
    h = min(src1.height, src2.height)
    w = min(src1.width, src2.width)

    # read only that intersection (light)
    r1 = src1.read(1, window=((0, h), (0, w))).astype("float32")
    r2 = src2.read(1, window=((0, h), (0, w))).astype("float32")

    nodata = src1.nodata

    # build mask in HR intersection grid
    mask = np.isfinite(r1)
    if nodata is not None:
        mask &= (r1 != nodata)

    diff = r1 - r2
    diff[~mask] = nodata

    profile = src1.profile
    profile.update(dtype="float32", height=h, width=w, transform=src1.transform)

    with rasterio.open(out_diff, "w", **profile) as dst:
        dst.write(diff.astype("float32"), 1)

    print("Saved in:", out_diff)

In [ ]:
# Quantify raster differences
with rasterio.open(out_diff) as srcd:
    diff = srcd.read(1).astype("float32")
    nodata = srcd.nodata

# Build valid mask
mask = np.isfinite(diff)
if nodata is not None:
    mask &= (diff != nodata)

d = diff[mask]

# Basic stats 
print(f"Mean diff: {np.mean(d):.3f}")
print(f"Median diff: {np.median(d):.3f}")
print(f"Std diff: {np.std(d):.3f}")

# Error-like metrics 
mae = float(np.mean(np.abs(d)))
rmse = float(np.sqrt(np.mean(d**2)))
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(d, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
ax.axvline(0, color='black', linewidth=1, linestyle='--', label='Sin diferencia')
ax.axvline(np.mean(d), color='red', linewidth=1, linestyle='-', label=f'Media: {np.mean(d):.2f} °C')
ax.axvline(np.median(d), color='orange', linewidth=1, linestyle='-', label=f'Mediana: {np.median(d):.2f} °C')

ax.set_xlabel('Diferencia HR − Global (°C)', fontsize=10)
ax.set_ylabel('Frecuencia', fontsize=10)
ax.set_title('Distribución de diferencias píxel a píxel (MRT)', fontsize=11)
ax.set_xlim(-30, 30)  # adjust according to your data
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Plot result 
with rasterio.open(out_diff) as src:
    diff = src.read(1).astype("float32")
    xmin, ymin, xmax, ymax = src.bounds


# Difference map
fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(diff, cmap='RdBu_r')
ax.axis('off')
ax.set_title('Diferencia entre predicciones MRT (HR − Global)', fontsize=11)
cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, shrink=0.8)
cb.set_label('Diferencia (°C)', fontsize=9)
cb.ax.tick_params(labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
thresholds = [0,1.0, 2.0, 5.0, 10.0]
fractions  = [float(np.mean(np.abs(d) > thr) * 100) for thr in thresholds]
labels     = [f'> {t} °C' for t in thresholds]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, fractions, color='steelblue', edgecolor='white', linewidth=0.5)

# Values above each bar
for bar, val in zip(bars, fractions):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val:.1f}%',
        ha='center', va='bottom', fontsize=10
    )

ax.set_xlabel('Umbral de diferencia absoluta', fontsize=10)
ax.set_ylabel('Píxeles por encima del umbral (%)', fontsize=10)
ax.set_title('Diferencia |HR − Global| por umbral', fontsize=11)
ax.set_ylim(0, max(fractions) * 1.2)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### Studying differences from shadow inputs: buildings and trees (HR vs global datasets)

Shadows were modelled with pybdshadow from building and tree heights, which come from different sources in each case:

| Layer | Dataset | Source | Original format and resolution | Rasterized layer compared here |
|---|---|---|---|---|
| Buildings | HR (`3.HR-data-bologna`) | local buildings with height (`bologne_buildings_height.json`) | vector (footprint polygons with height) | 0.5 m |
| Buildings | Global (`2.global-data-bologna`) | Global Building Atlas (`GBA_bologna.geojson`) | vector (footprint polygons with height) | 0.5 m |
| Trees | HR (`3.HR-data-bologna`) | local trees with height (`bologne_trees_height`) | vector (canopy polygons with height) | 0.5 m |
| Trees | Global (`2.global-data-bologna`) | Meta global canopy height (`gch_bologna`) | raster, 1 m (integer heights) | 1 m |

Here we compare the rasterized height layers of both sources, since they are the inputs whose differences propagate to the shadows and, therefore, to the MRT.

**Method**
- Both sources are brought to a common **1 m grid** (0.5 m rasters are aggregated 2×2: a pixel has an object if at least half of its sub-pixels do, with the mean height of those sub-pixels) and compared **inside the city boundary**, where shadows were kept.
- A pixel counts as object if its height is > 0 m for buildings and > 2 m for trees (same filter applied to the tree vectors before computing shadows).
- HR data are taken as reference.

**Metrics**
- *Cover*: area and share of the city covered by each source.
- *Spatial agreement*: IoU (intersection / union), precision (share of the global area that is also in HR), recall (share of the HR area that the global source detects), omission (HR area missed by global) and commission (global area not in HR).
- *Heights*, only where both sources have an object: mean bias (global − HR), MAE, RMSE and Pearson r.

In [ ]:
# Parameters of the comparison
res = 1.0                # common grid resolution (m)
min_height = {           # minimum height (m) for a pixel to count as object
    "buildings": 0,      # any building
    "trees": 2,          # same filter applied to tree vectors before computing shadows (2.4.2.2)
}
block_rows = 1000        # rows of the 1 m grid processed per block (memory control)

In [ ]:
def read_block_1m(src, row_off, n_rows, width):
    """Read a block on the common 1 m grid. 0.5 m rasters are aggregated 2x2:
    the pixel counts as object if >= half of its sub-pixels have a height, and the
    height is the mean of those sub-pixels. NaN and 0 are treated as 'no object'."""
    factor = int(round(res / src.res[0]))
    win = Window(0, row_off * factor, width * factor, n_rows * factor)
    arr = src.read(1, window=win).astype("float32")
    arr[~np.isfinite(arr) | (arr <= 0)] = np.nan
    if factor == 1:
        return arr
    blocks = arr.reshape(n_rows, factor, width, factor)
    present = np.isfinite(blocks)
    n_present = present.sum(axis=(1, 3))
    height_sum = np.where(present, blocks, 0).sum(axis=(1, 3))
    out = np.full((n_rows, width), np.nan, dtype="float32")
    keep = n_present >= (factor * factor) / 2
    out[keep] = height_sum[keep] / n_present[keep]
    return out


def compare_layer(path_hr, path_global, boundary, threshold):
    """Accumulate block by block: agreement counts and height errors where both
    sources have an object."""
    acc = {"city": 0, "both": 0, "hr_only": 0, "global_only": 0,
           "n": 0, "sum_d": 0.0, "sum_abs_d": 0.0, "sum_d2": 0.0,
           "sum_hr": 0.0, "sum_gl": 0.0, "sum_hr2": 0.0, "sum_gl2": 0.0, "sum_hrgl": 0.0}

    with rasterio.open(path_hr) as src_hr, rasterio.open(path_global) as src_gl:
        left, bottom, right, top = src_hr.bounds
        width = int(round((right - left) / res))
        height = int(round((top - bottom) / res))
        transform_1m = rasterio.transform.from_origin(left, top, res, res)
        geoms = boundary.to_crs(src_hr.crs).geometry

        for row_off in range(0, height, block_rows):
            n_rows = min(block_rows, height - row_off)
            block_transform = rasterio.windows.transform(Window(0, row_off, width, n_rows), transform_1m)
            in_city = geometry_mask(geoms, out_shape=(n_rows, width), transform=block_transform, invert=True)
            if not in_city.any():
                continue

            h_hr = read_block_1m(src_hr, row_off, n_rows, width)
            h_gl = read_block_1m(src_gl, row_off, n_rows, width)

            obj_hr = (h_hr > threshold) & in_city
            obj_gl = (h_gl > threshold) & in_city
            both = obj_hr & obj_gl

            acc["city"] += int(in_city.sum())
            acc["both"] += int(both.sum())
            acc["hr_only"] += int((obj_hr & ~obj_gl).sum())
            acc["global_only"] += int((obj_gl & ~obj_hr).sum())

            a, b = h_hr[both].astype("float64"), h_gl[both].astype("float64")
            d = b - a
            acc["n"] += a.size
            acc["sum_d"] += d.sum(); acc["sum_abs_d"] += np.abs(d).sum(); acc["sum_d2"] += (d ** 2).sum()
            acc["sum_hr"] += a.sum(); acc["sum_gl"] += b.sum()
            acc["sum_hr2"] += (a ** 2).sum(); acc["sum_gl2"] += (b ** 2).sum(); acc["sum_hrgl"] += (a * b).sum()

    return acc


def summarize(acc, layer):
    """Metrics from the accumulated counts (HR is taken as reference)."""
    px_area_km2 = res * res / 1e6
    hr = acc["both"] + acc["hr_only"]
    gl = acc["both"] + acc["global_only"]
    n = acc["n"]
    mean_hr, mean_gl = acc["sum_hr"] / n, acc["sum_gl"] / n
    cov = acc["sum_hrgl"] / n - mean_hr * mean_gl
    std_hr = np.sqrt(acc["sum_hr2"] / n - mean_hr ** 2)
    std_gl = np.sqrt(acc["sum_gl2"] / n - mean_gl ** 2)
    precision = acc["both"] / gl
    recall = acc["both"] / hr
    return {
        "capa": layer,
        "área HR (km²)": hr * px_area_km2,
        "área global (km²)": gl * px_area_km2,
        "cobertura HR (% ciudad)": hr / acc["city"] * 100,
        "cobertura global (% ciudad)": gl / acc["city"] * 100,
        "IoU": acc["both"] / (acc["both"] + acc["hr_only"] + acc["global_only"]),
        "precisión (global correcto)": precision,
        "recall (HR detectado)": recall,
        "omisión global (% área HR)": acc["hr_only"] / hr * 100,
        "comisión global (% área global)": acc["global_only"] / gl * 100,
        "altura media HR (m)": mean_hr,
        "altura media global (m)": mean_gl,
        "sesgo global − HR (m)": acc["sum_d"] / n,
        "MAE (m)": acc["sum_abs_d"] / n,
        "RMSE (m)": np.sqrt(acc["sum_d2"] / n),
        "r de Pearson": cov / (std_hr * std_gl),
    }

#### Metrics

In [ ]:
# Compare buildings and trees (HR vs global) on the common 1 m grid, inside the city boundary -- several minutes
boundary = gpd.read_file(city_boundary)

layers = {
    "buildings": (buildings_hr, buildings_global),
    "trees": (trees_hr, trees_global),
}

results, counts = [], {}
for layer, (path_hr, path_gl) in layers.items():
    acc = compare_layer(path_hr, path_gl, boundary, min_height[layer])
    results.append(summarize(acc, layer))
    counts[layer] = acc  # pixel counts, used in the plot below
    print(f"{layer}: done")

metrics = pd.DataFrame(results).set_index("capa").T
metrics.to_csv(out_shadow_inputs_csv)
metrics.round(3)

#### Cover and spatial agreement

In [ ]:
# Share of the area detected as each class (union of both sources)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
titles = {"buildings": "Edificios", "trees": "Árboles"}

for ax, layer in zip(axes, layers):
    detected = [counts[layer][k] for k in ("hr_only", "both", "global_only")]
    values = [n / sum(detected) * 100 for n in detected]  # % of the area detected by at least one source
    labels = ["Solo HR", "Ambas", "Solo global"]
    bars = ax.bar(labels, values, color=["tab:blue", "tab:green", "tab:orange"])
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{val:.1f} %", ha="center", va="bottom", fontsize=9)
    ax.set_title(titles[layer], fontsize=11)
    ax.set_ylabel("Superficie (%)")
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

#### Agreement map

In [ ]:
# Agreement map in a sample window (same window used for the MRT comparison above)
win_xmin, win_ymin, win_xmax, win_ymax = 686000, 4928000, 687000, 4929000

cmap = ListedColormap(["white", "tab:green", "tab:blue", "tab:orange"])
labels = ["Ninguna", "Ambas", "Solo HR", "Solo global"]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, layer in zip(axes, layers):
    path_hr, path_gl = layers[layer]
    present = []
    for path in (path_hr, path_gl):
        with rasterio.open(path) as src:
            win = from_bounds(win_xmin, win_ymin, win_xmax, win_ymax, src.transform).round_offsets().round_lengths()
            factor = int(round(res / src.res[0]))
            h = read_block_1m(src, int(win.row_off) // factor, int(win.height) // factor, src.width // factor)
            col0 = int(win.col_off) // factor
            h = h[:, col0:col0 + int(win.width) // factor]
        present.append(h > min_height[layer])
    obj_hr, obj_gl = present
    category = np.select([obj_hr & obj_gl, obj_hr & ~obj_gl, ~obj_hr & obj_gl], [1, 2, 3], default=0)

    ax.imshow(category, cmap=cmap, vmin=0, vmax=3, interpolation="nearest")
    ax.set_title(titles[layer], fontsize=11)
    ax.axis("off")

handles = [plt.Rectangle((0, 0), 1, 1, color=cmap(i), ec="grey") for i in range(4)]
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
plt.suptitle("Coincidencia entre fuentes (ventana de 1 km × 1 km)", fontsize=12)
plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.show()